# Bjerknes feedback changes over time

## imports

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import copy
import time

# Import custom modules
import src.utils

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Funcs

In [ ]:
def window(x):
    return src.utils.get_windowed(x, stride=120)

## Load data

#### $T$, $h$

In [ ]:
## open data
Th = src.utils.load_cesm_indices(load_z20=True)

### Spatial data

#### Load

In [ ]:
## load spatial data
forced, anom = src.utils.load_consolidated()

## add T,h information
anom = xr.merge([anom, Th])

#### scaling

In [ ]:
hbar_scale = xr.open_dataarray(
    pathlib.Path(SAVE_FP, "cesm_Hbar_scale_v2.nc"),
)

## Compute/plot

In [ ]:
def get_slope_bymonth(x, **kwargs):
    """get slope for each month separately"""
    return x.groupby("time.month").map(src.utils.regress_proj, **kwargs)

### Specify variabes

In [ ]:
## specify plot month
PLOT_MONTH = 1

## specify boxes for fitting
T_VAR = "T_3"
T_FN = src.utils.get_nino3
Tsub_FN = (
    lambda x, H: x.sel(longitude=slice(210, 270))
    .mean("longitude")
    .sel(z_t=H, method="nearest")
)

## specify subsetting funcs
LATS = dict(latitude=slice(-5, 5))
LATS_H = dict(latitude=slice(-5, 5))
LONS_E = dict(longitude=slice(210, 270))
LONS_W = dict(longitude=slice(120, 160))
LONS_TAU = dict(longitude=slice(150, 220))

Set funcs

In [ ]:
## helper func to select and avg
def sel_helper(x, lats, lons):
    """helper func to avg over lats/lons"""

    ## first, average over lons
    x_avg = x.sel(lons).mean("longitude")

    if "latitude" in x_avg.dims:
        x_avg = x_avg.sel(lats).mean("latitude")

    return x_avg


## specify functions
TAU_FN = lambda x: sel_helper(x, LATS, LONS_TAU)
He_FN = lambda x: sel_helper(x, LATS_H, LONS_E)
Hw_FN = lambda x: sel_helper(x, LATS_H, LONS_W)
Hgrad_FN = lambda x: He_FN(x) - Hw_FN(x)

#### helper funcs

In [ ]:
def regress_over_time(data, x_vars, y_vars):
    """regression over time"""

    ## get windowed data
    data_ = window(data)

    ## empty list to hold coefficients
    coefs = []

    ## shared args
    kwargs = dict(x_vars=x_vars, y_vars=y_vars)

    ## loop thru years
    for year in tqdm.tqdm(data_.year):

        ## get grouped data
        data_y = data_.sel(year=year).groupby("time.month")

        ## do regression
        coefs.append(data_y.map(src.utils.regress_xr_proj, **kwargs))

    return xr.concat(coefs, dim=data_.year)


def regress_wrapper(data, x_vars, y_var, y_fn):
    """regression over time"""

    ## prep data
    y_data = src.utils.reconstruct_wrapper(data[[f"{y_var}", f"{y_var}_comp"]], fn=y_fn)

    ## subset for data
    data_ = xr.merge([data[x_vars], y_data])

    return regress_over_time(data_, x_vars=x_vars, y_vars=[y_var])


def frac_change(x):
    """fractional change"""
    return x / x.isel(year=0) - 1

### SST-$\tau_x$

In [ ]:
mu_a = regress_wrapper(data=anom, x_vars=["T_3"], y_var="taux", y_fn=TAU_FN)
mu_a = mu_a["taux"].squeeze(drop=True).rename("mu_a")

### taux-SSH

In [ ]:
## get tau index
taux_wpac = src.utils.reconstruct_wrapper(anom[["taux", "taux_comp"]], fn=TAU_FN)
anom_ = xr.merge([taux_wpac, anom[["ssh", "ssh_comp"]]])

## regress (v1)
beta_h = regress_wrapper(data=anom_, x_vars=["taux"], y_var="ssh", y_fn=He_FN)
beta_h = beta_h["ssh"].squeeze(drop=True).rename("beta_h")

## regress (v2)
Hgrad_FN = lambda x: He_FN(x) - Hw_FN(x)
beta_h_ = regress_wrapper(data=anom_, x_vars=["taux"], y_var="ssh", y_fn=Hgrad_FN)
beta_h_ = beta_h_["ssh"].squeeze(drop=True).rename("beta_h")

Scatter plot

In [ ]:
## time indices
# t_idx = dict(time=slice(None,480))
t_idx = dict(time=slice(-480, None))
get_t = lambda x: x.isel(t_idx)

## prep func
get_djf = lambda x: src.utils.sel_month(x.resample({"time": "QS-DEC"}).mean(), 12)
prep = lambda x: get_djf(x).transpose("time", "member")

## get data
x = get_t(taux_wpac)["taux"]
h_e = src.utils.reconstruct_wrapper(
    get_t(anom[["ssh", "ssh_comp"]]),
    fn=He_FN,
)["ssh"]

h_w = src.utils.reconstruct_wrapper(
    get_t(anom[["ssh", "ssh_comp"]]),
    fn=Hw_FN,
)["ssh"]

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
ax.scatter(
    # prep(x), prep(h_e) - prep(h_w), s=3,
    prep(x),
    prep(h_e),
    s=3,
)

plt.show()
# ax.scatter(
#     prep(x), prep(h_e), s=3,
# )

### subsurface terms

In [ ]:
## specify mixed layer depth
H_, H0 = (0, 50)

## specify volume average
vol_avg = lambda x: x.sel(longitude=slice(210, 270), z_t=slice(H_, H0)).mean(
    ["longitude", "z_t"]
)

#### SSH-$T_{sub}$

In [ ]:
## get ssh index
ssh_epac = src.utils.reconstruct_wrapper(anom[["ssh", "ssh_comp"]], fn=He_FN)
anom_ = xr.merge([ssh_epac, anom[["T", "T_comp"]]])

## regress
a_h = regress_wrapper(data=anom_, x_vars=["ssh"], y_var="T", y_fn=vol_avg)
a_h = a_h["T"].squeeze(drop=True).rename("a_h")

#### $\overline{w}$

In [ ]:
## get w_bar
w_bar = src.utils.reconstruct_clim(window(forced[["w", "w_comp"]]))["w"]
is_downwelling = w_bar < 0

## only keep upwelling
w_bar_upwelling = w_bar.where(~(is_downwelling), other=0.0)

## average over Niño 3 and scale by MLD
w_H = 1 / H0 * vol_avg(w_bar_upwelling).rename("w_H")

#### Consolidate data

In [ ]:
## should we scale by mean thermocline depth?
scale_by_h = True

## which beta_h should we use?
beta_h_plot = beta_h_

## handle scaling
if scale_by_h:
    scale = hbar_scale
else:
    scale = 1

## apply scaling
beta_h_plot = (beta_h_plot * scale).rename("beta_h")
a_h_plot = (a_h / scale).rename("a_h")

## compute THF
THF = (mu_a * beta_h_plot * a_h_plot * w_H).rename("thf")

## put data in xarray
params = xr.merge([THF, mu_a, beta_h_plot, a_h_plot, w_H])
labels = [r"$\mu_a\beta_h a_h$", r"$\mu_a$", r"$\beta_h$", r"$a_h$", r"$w/H$"]
label_dict = dict(zip(list(params), labels))

Plot annual mean changes

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 2.75), layout="constrained")

## plot total
axs[0].plot(
    params.year,
    frac_change(params["thf"].mean("month")),
    c="k",
    label=label_dict["thf"],
)

# plot terms
for p in list(params)[1:]:
    axs[1].plot(params.year, frac_change(params[p].mean("month")), label=label_dict[p])

## format/labeling
src.utils.set_lims(axs)
axs[1].set_yticks([])
ax_kwargs = dict(c="k", lw=0.8, ls="--")
for ax in axs:
    ax.axhline(0, **ax_kwargs)
    ax.axvline(2030, **ax_kwargs)
    ax.set_xticks([1870, 2030])
    ax.legend()


plt.show()

In [ ]:
# fig,axs = plt.subplots(1,5,figsize=(2,10))

# for
plt.plot(THF.sel(year=1870))
plt.plot(THF.sel(year=2030))
plt.plot(THF.sel(year=2080))